# 07. 원인 확인용 제출 파일 만들기

### 무슨 일이 있었나
| 제출 | dayofyear | 과적합 방지 설정 | 우리 검증 | **데이콘 점수** |
|------|:--:|:--:|:--:|:--:|
| 1주차 베이스라인 | 있음 | 없음 | 0.556 | **0.468** ← 더 좋음 |
| 3주차 최종 | 제거 | 적용 | 0.534 | 0.485 |

우리 검증(기출)에서는 3주차가 좋았는데, 데이콘(진짜 시험)에서는 1주차가 더 좋았어요.

### 가장 의심되는 원인: dayofyear 제거
- 2주차에 dayofyear를 빼서 좋아진 건 **fold1뿐**이었어요 (fold2·3은 오히려 살짝 나빠짐).
- fold1이 좋아진 이유: fold1 모델은 **10~12월 날짜를 한 번도 못 봐서** dayofyear를 엉뚱하게 썼기 때문.
- 그런데 **제출용 최종 모델은 1년치 날짜를 다 봤어요** → fold1의 문제는 검증 때만 있는 문제(착시)였을 가능성.

### 이 노트북이 하는 일: 하나씩 바꿔서 제출 파일 만들기
1주차 베이스라인과 **딱 하나만 다르게** 만들어서 어느 변화가 점수를 떨어뜨렸는지 확인합니다.
1주차와 똑같이 **시드 1개(42)** 로 만들어서 조건을 맞춰요.

| 파일 | dayofyear | 과적합 방지 | 알 수 있는 것 | 예상 |
|------|:--:|:--:|------|------|
| `check1_doy_reg.csv` | 있음 | **적용** | 과적합 방지 설정만의 효과 | 0.468과 비슷하거나 좋음 |
| `check2_nodoy.csv` | **제거** | 없음 | dayofyear 제거만의 효과 | 0.468보다 나쁨 |

⚠️ **데이콘 점수 과적합 주의:** 데이콘 점수를 보고 계속 바꿔서 내면, 이번엔 데이콘 점수에 맞추는 과적합이 돼요.
**원인 확인용으로 이 2개만** 제출하고, 결과는 **우리 검증 방식을 고치는 데** 씁니다.

> ⏱️ 검증 없이 최종 모델 2개만 학습해서 **5~10분**이면 끝나요.

## 0. 환경 설정

In [ ]:
# ▶ Google Colab에서만 실행되는 셀입니다 (내 컴퓨터에서는 아무 일도 안 함).
#   구글 드라이브 연결 팝업이 뜨면 '허용'을 눌러주세요.
import sys
if 'google.colab' in sys.modules:
    !pip install -q koreanize-matplotlib
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
import os, sys, time, warnings
import platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50); pd.set_option('display.width', 200)
try:
    import koreanize_matplotlib
except ImportError:
    if platform.system() == 'Windows': plt.rc('font', family='Malgun Gothic')
    elif platform.system() == 'Darwin': plt.rc('font', family='AppleGothic')
plt.rc('axes', unicode_minus=False)

IN_COLAB = 'google.colab' in sys.modules
BASE_DIR = Path('/content/drive/MyDrive/capstone') if IN_COLAB else Path('..')
DATA_DIR = Path(os.environ.get('DATA_DIR', BASE_DIR / 'data'))
FIG_DIR = Path(os.environ.get('FIG_DIR', BASE_DIR / 'figures'))
RESULT_DIR = Path(os.environ.get('RESULT_DIR', BASE_DIR / 'results'))
for d in [FIG_DIR, RESULT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print('LightGBM', lgb.__version__, '| train.csv 존재 여부:', (DATA_DIR / 'train' / 'train.csv').exists())

import re
SUB_DIR = Path(os.environ.get('SUB_DIR', BASE_DIR / 'submissions')); SUB_DIR.mkdir(parents=True, exist_ok=True)

## 1. 데이터 준비 (3주차와 동일)

In [ ]:
KR_HOLIDAYS = pd.to_datetime([
    '2023-01-01','2023-01-21','2023-01-22','2023-01-23','2023-01-24','2023-03-01','2023-05-05','2023-05-27',
    '2023-05-29','2023-06-06','2023-08-15','2023-09-28','2023-09-29','2023-09-30','2023-10-02','2023-10-03',
    '2023-10-09','2023-12-25','2024-01-01','2024-02-09','2024-02-10','2024-02-11','2024-02-12','2024-03-01',
    '2024-04-10','2024-05-05','2024-05-06','2024-05-15','2024-06-06',
    # TEST 기간(2024-06 ~ 2025-06) 공휴일
    '2024-08-15','2024-09-16','2024-09-17','2024-09-18','2024-10-01','2024-10-03','2024-10-09','2024-12-25',
    '2025-01-01','2025-01-27','2025-01-28','2025-01-29','2025-01-30','2025-03-01','2025-03-03','2025-05-05',
    '2025-05-06','2025-06-03','2025-06-06'])
HOLIDAY_SET = set(KR_HOLIDAYS)

train = pd.read_csv(DATA_DIR / 'train' / 'train.csv')
train['영업일자'] = pd.to_datetime(train['영업일자'])
train['매출수량'] = train['매출수량'].clip(lower=0)
wide = (train.pivot_table(index='영업일자', columns='영업장명_메뉴명', values='매출수량', aggfunc='sum')
             .sort_index().asfreq('D').fillna(0))
MENUS = wide.columns.tolist()
VENUE_OF = {m: m.split('_', 1)[0] for m in MENUS}
VENUES = sorted(set(VENUE_OF.values()))
MENU_VENUE = np.array([VENUE_OF[m] for m in MENUS])
VENUE_IDX = np.array([VENUES.index(v) for v in MENU_VENUE])      # 메뉴 → 영업장 번호
IS_GROUP = np.array(['단체' in m for m in MENUS]).astype(int)       # 단체 메뉴 표시
W = wide.values.astype(np.float32)
DATES = wide.index
M, V = len(MENUS), len(VENUES)
INPUT_LEN, HORIZON = 28, 7
FOLDS = [('2023-10-01', '2023-12-31'), ('2024-01-01', '2024-03-31'), ('2024-04-01', '2024-06-09')]
print('메뉴', M, '| 영업장', V, '| 단체 메뉴 수', IS_GROUP.sum())

def smape_1d(y_true, y_pred):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    yt, yp = y_true[mask], y_pred[mask]
    return np.mean(2 * np.abs(yt - yp) / (np.abs(yt) + np.abs(yp)))

def venue_smape(T, P):
    scores = {}
    for v in VENUES:
        idx = np.where(MENU_VENUE == v)[0]
        s = [smape_1d(T[:, i], P[:, i]) for i in idx]
        s = [x for x in s if not np.isnan(x)]
        if s:
            scores[v] = float(np.mean(s))
    return float(np.mean(list(scores.values()))), scores

def fold_origins(val_start, val_end):
    vs, ve = DATES.get_loc(pd.Timestamp(val_start)), DATES.get_loc(pd.Timestamp(val_end))
    return list(range(INPUT_LEN, vs - HORIZON + 1)), list(range(vs, ve - HORIZON + 2, 7))

## 2. 피처 만들기 (3주차와 동일 + dayofyear 다시 포함)

In [ ]:
MENU_CAT = pd.CategoricalDtype(MENUS); VENUE_CAT = pd.CategoricalDtype(VENUES)
GROUPS = {
    'A_추세': ['trend_7_28', 'trend_7_prev7', 'trend_sold_7_28'],
    'B_영업장흐름': ['venue_mean_7', 'venue_mean_28', 'venue_trend_7_28', 'venue_active_ratio_7', 'menu_share_28'],
    'C_영업장상태': ['venue_closed_7', 'venue_zero_days_28', 'venue_days_since_reopen'],
    'D_단체메뉴': ['is_group'],
}
ALL_NEW = sum(GROUPS.values(), [])

def venue_status(vx):
    """vx: (28, 영업장 수) 영업장별 하루 총판매 → 영업장별 상태 피처 3개"""
    closed7 = (vx[-7:].sum(0) == 0).astype(int)
    zero_days = (vx == 0).sum(0)
    since = np.full(vx.shape[1], INPUT_LEN + 1)
    for j in range(vx.shape[1]):
        z = vx[:, j] == 0
        run, end_of_long = 0, None                    # 7일 이상 연속 0이 끝난 위치 찾기
        for t in range(INPUT_LEN):
            run = run + 1 if z[t] else 0
            if run >= 7 and (t == INPUT_LEN - 1 or not z[t + 1]):
                end_of_long = t
        if end_of_long is not None:
            since[j] = 0 if end_of_long == INPUT_LEN - 1 else INPUT_LEN - 1 - end_of_long
    return closed7, zero_days, since

def make_features(inp, start_date):
    """inp: (28, 메뉴 수). 반환: 7 × 메뉴 수 행"""
    x = np.asarray(inp, dtype=np.float32).T                      # (메뉴, 28)
    vx = np.stack([x[VENUE_IDX == j].sum(0) for j in range(V)], 1)  # (28, 영업장)
    F = {f'lag_{k}': x[:, INPUT_LEN - k] for k in range(1, INPUT_LEN + 1)}
    for w in (7, 14, 28):
        y = x[:, -w:]; cnt = (y > 0).sum(1)
        F[f'mean_{w}'] = y.mean(1)
        F[f'sold_mean_{w}'] = np.where(cnt > 0, y.sum(1) / np.maximum(cnt, 1), 0)
        F[f'sold_ratio_{w}'] = cnt / w
    with np.errstate(all='ignore'):
        F['sold_median_28'] = np.nan_to_num(np.nanmedian(np.where(x > 0, x, np.nan), axis=1))
    F['std_28'] = x.std(1); F['max_28'] = x.max(1)
    F['days_since_sale'] = np.array([INPUT_LEN - np.nonzero(r)[0][-1] if r.any() else INPUT_LEN + 1 for r in x])
    # A. 추세
    F['trend_7_28'] = (F['mean_7'] + 1) / (F['mean_28'] + 1)
    F['trend_7_prev7'] = (x[:, -7:].sum(1) + 1) / (x[:, -14:-7].sum(1) + 1)
    F['trend_sold_7_28'] = (F['sold_mean_7'] + 1) / (F['sold_mean_28'] + 1)
    # B. 영업장 흐름
    v7, v28 = vx[-7:].mean(0), vx.mean(0)
    n_menu_v = np.bincount(VENUE_IDX, minlength=V)
    active7 = np.bincount(VENUE_IDX, weights=(x[:, -7:].sum(1) > 0), minlength=V) / n_menu_v
    F['venue_mean_7'] = v7[VENUE_IDX]; F['venue_mean_28'] = v28[VENUE_IDX]
    F['venue_trend_7_28'] = ((v7 + 1) / (v28 + 1))[VENUE_IDX]
    F['venue_active_ratio_7'] = active7[VENUE_IDX]
    F['menu_share_28'] = x.sum(1) / np.maximum(vx.sum(0)[VENUE_IDX], 1)
    # C. 영업장 상태
    closed7, zero_days, since = venue_status(vx)
    F['venue_closed_7'] = closed7[VENUE_IDX]; F['venue_zero_days_28'] = zero_days[VENUE_IDX]
    F['venue_days_since_reopen'] = since[VENUE_IDX]
    # D. 단체 메뉴
    F['is_group'] = IS_GROUP
    base = pd.DataFrame(F)
    base['menu'] = pd.Categorical(MENUS, dtype=MENU_CAT)
    base['venue'] = pd.Categorical(list(MENU_VENUE), dtype=VENUE_CAT)
    frames = []
    for h in range(1, HORIZON + 1):
        d = start_date + pd.Timedelta(days=h - 1); f = base.copy()
        f['h'] = h; f['dow'] = d.dayofweek; f['month'] = d.month; f['dayofyear'] = d.dayofyear
        f['is_weekend'] = int(d.dayofweek >= 5); f['is_holiday'] = int(d in HOLIDAY_SET)
        idx = [h - 1, h + 6, h + 13, h + 20]
        f['same_dow_last'] = x[:, h + 20]; f['same_dow_mean4'] = x[:, idx].mean(1)
        f['same_dow_sold_ratio'] = (x[:, idx] > 0).mean(1)
        frames.append(f)
    return pd.concat(frames, ignore_index=True)

def build_dataset(origins):
    X = pd.concat([make_features(W[o - INPUT_LEN:o], DATES[o]) for o in origins], ignore_index=True)
    y = np.concatenate([W[o:o + HORIZON].reshape(-1) for o in origins])
    return X, y

BASE_COLS = [c for c in make_features(W[:INPUT_LEN], DATES[INPUT_LEN]).columns if c not in ALL_NEW]
print('기존 피처 수:', len(BASE_COLS), '| 새 피처 수:', len(ALL_NEW))

## 3. train 전체로 학습 데이터 만들기

In [ ]:
t0 = time.time()
all_o = list(range(INPUT_LEN, len(DATES) - HORIZON + 1))
X_all, y_all = build_dataset(all_o)
mask = y_all > 0                                   # 판매일만 학습 (1주차와 동일)
COLS_DOY = BASE_COLS                               # 1주차 피처 (dayofyear 포함)
COLS_NODOY = [c for c in BASE_COLS if c != 'dayofyear']
print(f'준비 완료 ({time.time() - t0:.0f}초) | 피처 수: dayofyear 포함 {len(COLS_DOY)}, 제외 {len(COLS_NODOY)}')

## 4. 두 가지 설정으로 학습 & 제출 파일 저장
| 설정 | 내용 |
|------|------|
| 1주차 설정 | num_leaves 63, min_child_samples 50, colsample 0.8 |
| 과적합 방지 설정 | num_leaves 31, min_child_samples 100, reg_lambda 1.0, colsample 0.7 |

In [ ]:
PARAMS_W1 = dict(objective='regression', learning_rate=0.05, n_estimators=600,
                 num_leaves=63, min_child_samples=50,
                 subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1)
PARAMS_REG = {**PARAMS_W1, 'num_leaves': 31, 'min_child_samples': 100, 'reg_lambda': 1.0, 'colsample_bytree': 0.7}

CONFIGS = {
    'check1_doy_reg': (COLS_DOY, PARAMS_REG),     # dayofyear 있음 + 과적합 방지
    'check2_nodoy':   (COLS_NODOY, PARAMS_W1),    # dayofyear 제거 + 1주차 설정
}

sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')
cols_sub = [m for m in MENUS if m in sample.columns]
test_inputs = []
for f in sorted((DATA_DIR / 'test').glob('*TEST_*.csv')):
    tid = re.search(r'TEST_\d+', f.name).group()
    t = pd.read_csv(f); t['영업일자'] = pd.to_datetime(t['영업일자']); t['매출수량'] = t['매출수량'].clip(lower=0)
    tw = (t.pivot_table(index='영업일자', columns='영업장명_메뉴명', values='매출수량', aggfunc='sum')
            .sort_index().asfreq('D').reindex(columns=MENUS).fillna(0)).iloc[-INPUT_LEN:]
    test_inputs.append((tid, make_features(tw.values, tw.index.max() + pd.Timedelta(days=1))))

for name, (cols, params) in CONFIGS.items():
    t0 = time.time()
    model = lgb.LGBMRegressor(**params, random_state=42)
    model.fit(X_all.loc[mask, cols], np.log1p(y_all[mask]))
    sub = sample.set_index('영업일자').astype(float)
    for tid, Xt in test_inputs:
        pred = np.clip(np.expm1(model.predict(Xt[cols])), 0, None).reshape(HORIZON, M)
        for h in range(1, HORIZON + 1):
            sub.loc[f'{tid}+{h}일', cols_sub] = pd.Series(pred[h - 1], index=MENUS)[cols_sub].values
    sub.reset_index().to_csv(SUB_DIR / f'{name}.csv', index=False, encoding='utf-8-sig')
    print(f'{name}.csv 저장 완료 ({time.time() - t0:.0f}초) | 예측 평균 {sub[cols_sub].values.mean():.2f}')

## 5. 제출하고 결과 적기 ✍️
드라이브 `capstone/submissions/` 폴더의 두 파일을 다운로드해서 데이콘에 제출하고, 점수를 적어 주세요.

| 제출 | dayofyear | 과적합 방지 | 데이콘 점수 |
|------|:--:|:--:|:--:|
| 1주차 베이스라인 | 있음 | 없음 | 0.4679 |
| check1_doy_reg | 있음 | 적용 | |
| check2_nodoy | 제거 | 없음 | |
| 3주차 최종 | 제거 | 적용 | 0.4848 |

**읽는 법**
- check2가 나쁘고 check1이 좋거나 비슷 → **dayofyear 제거가 원인** (예상대로)
- check1도 나쁨 → 과적합 방지 설정도 실제 시험에는 안 맞았던 것
- 둘 다 0.468과 비슷 → 두 변화가 **함께** 있을 때만 나빠지는 것 (또는 시드 차이)